In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers datasets accelerate
!pip install -q torch
!pip install -q peft
!pip install -q einops
!pip install -q tqdm

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "EleutherAI/pythia-160m"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_hidden_states=True
)

model.eval()

In [ ]:
print(model)

In [ ]:
print(len(model.gpt_neox.layers))

In [ ]:
import torch

text = "Python functions are useful for machine learning."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states

print(len(hidden_states))

In [ ]:
layer6 = hidden_states[6]

print(layer6.shape)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train"
)

print(dataset)

In [ ]:
print(dataset)

In [ ]:
print(dataset[0])

In [ ]:
import torch
from tqdm import tqdm

def collect_activations(
    dataset,
    model,
    tokenizer,
    target_layer=6,
    num_samples=5000,
    max_length=128
):

    activations = []

    model.eval()

    for i in tqdm(range(num_samples)):

        text = dataset[i]["text"]

        if len(text.strip()) == 0:
            continue

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        inputs = {
            k: v.cuda()
            for k, v in inputs.items()
        }

        with torch.no_grad():

            outputs = model(**inputs)

        layer = outputs.hidden_states[target_layer]

        layer = layer.squeeze(0)

        activations.append(
            layer.cpu()
        )

    return torch.cat(
        activations,
        dim=0
    )

In [ ]:
device = "cuda"

model = model.to(device)

In [ ]:
print(next(model.parameters()).device)

In [ ]:
activations = collect_activations(
    dataset,
    model,
    tokenizer,
    target_layer=6,
    num_samples=5000
)

print(activations.shape)

In [ ]:
print(activations[0][:10])

In [ ]:
torch.save(
    activations,
    "/content/drive/MyDrive/layer6_activations.pt"
)

In [ ]:
import os

size_mb = os.path.getsize(
    "/content/drive/MyDrive/layer6_activations.pt"
)/(1024**2)

print(size_mb)

In [ ]:
loaded = torch.load(
    "/content/drive/MyDrive/layer6_activations.pt"
)

print(loaded.shape)

In [ ]:
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

dataset = TensorDataset(
    activations.float()
)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True
)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class SparseAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim=768,
        dict_size=6144,
        k=30
    ):
        super().__init__()

        self.k = k

        self.encoder = nn.Linear(
            input_dim,
            dict_size
        )

        self.decoder = nn.Linear(
            dict_size,
            input_dim,
            bias=False
        )

        self.pre_bias = nn.Parameter(
            torch.zeros(input_dim)
        )

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder.weight.data.norm(
            dim=0, keepdim=True
        )
        self.decoder.weight.data = (
            self.decoder.weight.data
            / norms.clamp(min=1e-8)
        )

    def forward(self, x):

        x_centered = x - self.pre_bias

        pre_activations = self.encoder(x_centered)


        topk_vals, topk_idx = torch.topk(
            pre_activations, self.k, dim=-1
        )

        features = torch.zeros_like(pre_activations)
        features.scatter_(-1, topk_idx, torch.relu(topk_vals))

        reconstruction = (
            self.decoder(features)
            + self.pre_bias
        )

        return reconstruction, features

In [ ]:
device = "cuda"

sae = SparseAutoencoder(
    input_dim=768,
    dict_size=6144
).to(device)

In [ ]:
def sae_loss(
    x,
    reconstruction,
    features,
    sparsity_weight=0

    recon_loss = (
        (x - reconstruction) ** 2
    ).mean()

    total_loss = recon_loss

    return (
        total_loss,
        recon_loss,
        torch.tensor(0.0)
    )

In [ ]:
activations = activations.float()

activations = activations.to(device)

In [ ]:
sae = SparseAutoencoder(
    input_dim=768,
    dict_size=6144,
    k=30
).to(device)

optimizer = torch.optim.Adam(
    sae.parameters(),
    lr=1e-4
)

scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=500
)

In [ ]:
epochs = 10
global_step = 0

for epoch in range(epochs):

    total_loss = 0
    total_recon = 0

    for batch in loader:

        x = batch[0].to(device)

        reconstruction, features = sae(x)

        loss, recon_loss, _ = sae_loss(x, reconstruction, features)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sae.normalize_decoder()
        scheduler.step()

        global_step += 1

        total_loss += loss.item()
        total_recon += recon_loss.item()

    print(
        f"Epoch {epoch+1} | "
        f"Loss={total_loss/len(loader):.4f} | "
        f"Recon={total_recon/len(loader):.4f}"
    )

In [ ]:
with torch.no_grad():
    sample = activations[:1000].float().to(device)
    _, features = sae(sample)
    active_fraction = (features > 0).float().mean()
    print("Active Fraction:", active_fraction.item())
    print("Expected:", 30/6144)  # HAD TRAINED SAE AGAIN ON 10 EPOCHS TO CONVERGE LOSS

In [ ]:
torch.save(
    sae.state_dict(),
    "/content/drive/MyDrive/sae_layer6_base.pt"
)
print("Saved.")